In [2]:
import struct
import numpy as np
import math
from numpy.random import *
import WinoTran_NCHW as NCHW

In [3]:
#generation
# chn = 512
# numOfFilter =512
# # print(224*224*64)
# # print(112*112*128)
# # print(56*56*256)
# # print(28*28*512)
# parameter = chn *3*3* numOfFilter

# input1 = (np.array(rand(parameter))-0.5).astype(np.float32)
# des = open("kernel.bin","wb")
# cnt = des.write(input1)
# des.close()

In [43]:
bat4Conv =1
inside = 224
chn = 64
numOfFilter =64
padding =1
inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)
M = (int)(blockn * blockn * bat4Conv);
N = numOfFilter;
K = chn

MSize = M if (M%128 == 0) else math.ceil(M/128)*128
NSize = N if (N%128 == 0) else math.ceil(N/128)*128
KSize = (int((K-1)/8)+1)*8

parameters1 = bat4Conv * inside * inside * chn
parameters2 = numOfFilter * 3 * 3 * chn

# readin the feature map
src = open("../../M1/data/input.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_input = input.reshape((bat4Conv,chn,inside,inside)).astype(np.float32)

inputTran1 = NCHW.Wino_inputTran(sample_input,padding)

# readin the filter data
src = open("../../M2/data/kernel.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_kernel = input.reshape((numOfFilter,chn,3,3)).astype(np.float32)
print(sample_kernel.shape)

kernelTran1 = NCHW.Wino_kernelTran(sample_kernel)
pyout1 = NCHW.gemm(inputTran, kernelTran, MSize, NSize, False)

(64, 64, 3, 3)


In [87]:
bat4Conv =1
inside = 14
chn = 512
numOfFilter =512
padding =1
inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)
M = (int)(blockn * blockn * bat4Conv);
N = numOfFilter;
K = chn

MSize = M if (M%128 == 0) else math.ceil(M/128)*128
NSize = N if (N%128 == 0) else math.ceil(N/128)*128
KSize = (int((K-1)/8)+1)*8

# readin the feature map
parameters1 = 36*MSize*KSize
src = open("../../M1/data/tc9/M1_new0.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
inputTran = input.reshape((36,KSize, MSize)).astype(np.float32)

# inputTran = NCHW.Wino_inputTran(sample_input,padding)


parameters2 = 36*NSize*KSize
# readin the filter data
src = open("../../M2/data/tc9/M2_new0.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
kernelTran = input.reshape((36,KSize, NSize)).astype(np.float32)
# print(sample_kernel.shape)

# kernelTran = NCHW.Wino_kernelTran(sample_kernel)

pyout2 = NCHW.gemm(inputTran, kernelTran, MSize, NSize, False)

In [88]:
for i in range(0,1):
    parameter2 = 36* MSize*NSize
    print(MSize,NSize)
    src = open("./tc9/M3_new0.bin","rb")
    context = src.read(parameter2*4)
    real_context = struct.unpack(str(parameter2)+'f',context)
    input = np.array(real_context)
    # input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
    testoutput = input.reshape((36,MSize,NSize)).astype(np.float32)
    print( np.sum(np.abs(pyout2-testoutput)) )

128 512
0.58604133


In [22]:
# readin the feature map
parameters1 = 36*MSize*KSize
src = open("../../M1/data/tc1/M1_orig.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
inputTran = input.reshape((36,KSize, MSize)).astype(np.float32)

# inputTran = NCHW.Wino_inputTran(sample_input,padding)


parameters2 = 36*NSize*KSize
# readin the filter data
src = open("../../M2/data/tc1/M2_new0.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
kernelTran = input.reshape((36,KSize, NSize)).astype(np.float32)
# print(sample_kernel.shape)

# kernelTran = NCHW.Wino_kernelTran(sample_kernel)

pyout = NCHW.gemm(inputTran, kernelTran, MSize, NSize, False)

In [68]:
print(pyout.shape)
print(pyout2)

(36, 3200, 128)
[[[ 3.1646070e-01 -1.2088400e-01 -2.6234919e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 5.3686804e-01 -3.9982080e-01  4.2124668e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 6.4281762e-01 -4.4530880e-01  1.0277152e-02 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  ...
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]]

 [[-2.6205871e-01  3.7069395e-02 -4.0091652e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 6.0808825e-01  1.2256195e-01 -8.9967960e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [-2.1267305e-01  4.4574025e-01  2.5000763e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  ...
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.000000

In [67]:
print(testoutput)

[[[-1.4194322e-01  1.1563764e+00 -1.3029853e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 2.1349056e+00 -3.2600796e+00 -9.4056988e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [-8.4486175e-01  2.8870537e+00  8.4140897e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  ...
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]]

 [[ 8.2975703e-01 -3.1923223e+00 -6.5738529e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [-1.3779917e+00  1.0269880e+00 -9.4939601e-01 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  [-1.7344518e+00 -9.1849387e-01 -3.3381033e+00 ...  0.0000000e+00
    0.0000000e+00  0.0000000e+00]
  ...
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 ...  0.0000000e+00
    0.0000

In [42]:
print( np.sum(np.abs(pyout1-pyout2)) )

0.0
